In [1]:
import json
from pathlib import Path
import re
import time

from tqdm.auto import tqdm

from schema_discovery import (
    load_prompt,
    analyze_snapshot,
    process_response,
)
from data_snapshot.utils import load_json

# Inputs

In [2]:
# Inputs
SYSTEM_PROMPT_PATH = "system_message.md"
USER_PROMPT_PATH = "user_message.md"
MODEL_NAME = "gpt-5.4-mini"
SLEEP_SECONDS = 0.2  # safety throttle
SOURCE = "refugee"  # "unhcr" or "prwp" or "refugee"
N_SAMPLES = 1  # None if processing all

SNAPSHOTS_DIR = f"/home/ajd/hf_datasets/data-snapshot/snapshots/{SOURCE}/"
METADATA_DIR = f"/home/ajd/hf_datasets/data-snapshot/metadata/{SOURCE}/"
OUTPUT_PATH = f"data/results_{SOURCE.lower()}.jsonl"

# Load data

In [3]:
system_prompt = load_prompt(SYSTEM_PROMPT_PATH)
user_prompt_template = load_prompt(USER_PROMPT_PATH)

In [4]:
metadata_files = list(Path(METADATA_DIR).glob("*.json"))
metadata_lookup = {x.stem: x for x in metadata_files}
len(metadata_files)

38

In [5]:
snapshot_files = list(Path(SNAPSHOTS_DIR).glob("*.png"))
len(snapshot_files)

499

In [6]:
# Sampling
if N_SAMPLES:
    import random
    figures = [x for x in snapshot_files if x.stem.split("_")[-2] == "figure"]
    tables = [x for x in snapshot_files if x.stem.split("_")[-2] == "table"]

    figures = random.sample(figures, N_SAMPLES)
    tables = random.sample(tables, N_SAMPLES)
    
    snapshot_files = figures + tables

len(snapshot_files)

2

# Main pipeline

In [7]:
# Build skip list
to_skip = set()
if Path(OUTPUT_PATH).exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            to_skip.add(json.loads(line)["snapshot_id"])

In [8]:
# Initialize dir
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for s in tqdm(snapshot_files):
        snapshot_id = s.name

        # Skip if in skip list
        if snapshot_id in to_skip:
            print(f"Skipped: {s.name}")
            continue

        # Base row — every row has the same keys
        row = {
            "snapshot_id": snapshot_id,
            "source": SOURCE,
            "parsed_output": None,
            "raw_output": None,
            "usage": None,
            "cost": None,
            "model": None,
            "error": None,
        }

        try:
            # Infer metadata file from snapshot filename
            parts = re.split("(_figure|_table)", s.stem)
            fname = parts[0]
            label = parts[1].lstrip("_")
            metadata_path = metadata_lookup.get(fname)
            if metadata_path:
                metadata = load_json(metadata_path)

            # Create user prompt
            user_prompt = user_prompt_template.replace(
                "{DOCUMENT_METADATA_JSON}",
                json.dumps(metadata, indent=2, ensure_ascii=False),
            )

            # Responses API
            response = analyze_snapshot(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                image_path=str(s),
                model=MODEL_NAME,
            )

            # Parse and process response
            result = process_response(response, model=MODEL_NAME)

            # Log data
            row["parsed_output"] = result["parsed_output"]
            row["raw_output"] = result["raw_output"]
            row["usage"] = result["usage"].model_dump()
            row["cost"] = result["cost"]
            row["model"] = MODEL_NAME
            row["error"] = result["error"]  # None on success, message on parse failure

        except Exception as e:
            row["error"] = str(e)

        # Single write point — always executes
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

        time.sleep(SLEEP_SECONDS)

  0%|          | 0/2 [00:00<?, ?it/s]